In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/Cleaned_data.csv")

In [3]:
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

In [4]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

INPUT_FILE = DATA_DIR / "Cleaned_data.csv"
ENGINEERED_FILE = DATA_DIR / "engineered_data.csv"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Input File:", INPUT_FILE)
print("Figures Folder:", FIGURES_DIR)

Project Root: c:\Users\Aditya Gupta\OneDrive\Desktop\Primeor_Solution
Input File: c:\Users\Aditya Gupta\OneDrive\Desktop\Primeor_Solution\data\Cleaned_data.csv
Figures Folder: c:\Users\Aditya Gupta\OneDrive\Desktop\Primeor_Solution\reports\figures


In [5]:
df.head(3)

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,profit_estimated
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.00,106.14,35.46,Medium,2011,False
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,Apac,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.10,36.04,9.72,Medium,2011,False
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,Emea,Emea,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.00,29.64,8.17,High,2011,False


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51271 entries, 0 to 51270
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          51271 non-null  object 
 1   order_date        51271 non-null  object 
 2   ship_date         51271 non-null  object 
 3   ship_mode         51271 non-null  object 
 4   customer_name     51271 non-null  object 
 5   segment           51271 non-null  object 
 6   state             51271 non-null  object 
 7   country           51271 non-null  object 
 8   market            51271 non-null  object 
 9   region            51271 non-null  object 
 10  product_id        51271 non-null  object 
 11  category          51271 non-null  object 
 12  sub_category      51271 non-null  object 
 13  product_name      51271 non-null  object 
 14  sales             51271 non-null  int64  
 15  quantity          51271 non-null  int64  
 16  discount          51271 non-null  float6

In [7]:
missing_values = df.isnull().sum()

missing_values[missing_values > 0]

Series([], dtype: int64)

### Feature Engineering

In [8]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])

df["shipping_days"] = (
    df["ship_date"] - df["order_date"]
).dt.days

df[["order_date", "ship_date", "shipping_days"]].head()

,order_date,ship_date,shipping_days
0,2011-01-01,2011-01-06,5
1,2011-01-01,2011-01-08,7
2,2011-01-01,2011-01-05,4
3,2011-01-01,2011-01-05,4
4,2011-01-01,2011-01-08,7


In [9]:
print("Minimum shipping days:", df["shipping_days"].min())
print("Maximum shipping days:", df["shipping_days"].max())
print("Average shipping days:", df["shipping_days"].mean())
print("Median shipping days:", df["shipping_days"].median())

Minimum shipping days: 0
Maximum shipping days: 7
Average shipping days: 3.969339392639114
Median shipping days: 4.0


In [10]:
negative_shipping = df[df["shipping_days"] < 0]

print("Negative shipping-day records:", len(negative_shipping))

Negative shipping-day records: 0


In [11]:
df["profit_margin"] = np.where(
    df["sales"] != 0,
    (df["profit"] / df["sales"]) * 100,
    np.nan
)

df[["sales", "profit", "profit_margin"]].head()

,sales,profit,profit_margin
0,408,106.14,26.01
1,120,36.04,30.03
2,66,29.64,44.91
3,45,-26.05,-57.90
4,114,37.77,33.13


In [12]:
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)

df[["order_date", "year_month"]].head()

,order_date,year_month
0,2011-01-01,2011-01
1,2011-01-01,2011-01
2,2011-01-01,2011-01
3,2011-01-01,2011-01
4,2011-01-01,2011-01


In [13]:
df["quarter"] = "Q" + df["order_date"].dt.quarter.astype(str)

df[["order_date", "quarter"]].head()

,order_date,quarter
0,2011-01-01,Q1
1,2011-01-01,Q1
2,2011-01-01,Q1
3,2011-01-01,Q1
4,2011-01-01,Q1


In [14]:
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.month_name()

df[
    ["order_date", "year", "month", "month_name", "year_month", "quarter"]
].head()

,order_date,year,month,month_name,year_month,quarter
0,2011-01-01,2011,1,January,2011-01,Q1
1,2011-01-01,2011,1,January,2011-01,Q1
2,2011-01-01,2011,1,January,2011-01,Q1
3,2011-01-01,2011,1,January,2011-01,Q1
4,2011-01-01,2011,1,January,2011-01,Q1


In [15]:
df["year"] = df["order_date"].dt.year
df["month"] = df["order_date"].dt.month
df["month_name"] = df["order_date"].dt.month_name()

df[
    ["order_date", "year", "month", "month_name", "year_month", "quarter"]
].head()

,order_date,year,month,month_name,year_month,quarter
0,2011-01-01,2011,1,January,2011-01,Q1
1,2011-01-01,2011,1,January,2011-01,Q1
2,2011-01-01,2011,1,January,2011-01,Q1
3,2011-01-01,2011,1,January,2011-01,Q1
4,2011-01-01,2011,1,January,2011-01,Q1


In [16]:
engineered_columns = [
    "shipping_days",
    "profit_margin",
    "year_month",
    "quarter",
    "year",
    "month",
    "month_name"
]

df[engineered_columns].head(10)

,shipping_days,profit_margin,year_month,quarter,year,month,month_name
0,5,26.01,2011-01,Q1,2011,1,January
1,7,30.03,2011-01,Q1,2011,1,January
2,4,44.91,2011-01,Q1,2011,1,January
3,4,-57.90,2011-01,Q1,2011,1,January
4,7,33.13,2011-01,Q1,2011,1,January
5,7,27.89,2011-01,Q1,2011,1,January
6,4,0.99,2011-01,Q1,2011,1,January
7,0,40.00,2011-01,Q1,2011,1,January
8,6,-35.03,2011-01,Q1,2011,1,January
9,4,37.98,2011-01,Q1,2011,1,January


In [17]:
df.to_csv(ENGINEERED_FILE, index=False)

print(f"Engineered dataset saved to: {ENGINEERED_FILE}")
print("Shape:", df.shape)

Engineered dataset saved to: c:\Users\Aditya Gupta\OneDrive\Desktop\Primeor_Solution\data\engineered_data.csv
Shape: (51271, 28)
